# 技能2 · Day 4 上机：企业级架构参考设计 + 行动研究

**版本**：v5.0 学习材料包（技能2收官）
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **pydantic** 设计CDP核心schema（Identity/Event/Segment/Profile四层），基于Segment Spec真实公开规范
2. 用 **networkx + TOGAF/ArchiMate** 建模企业AI架构四层依赖图，分析关键依赖路径
3. 用 **pandas** 分析行动研究（Action Research）迭代循环KPI，理解"研究即干预"
4. 把Day1-3整合为**营销中心AI原生参考架构**，定位为DSR artifact
5. 理解**天道推演×企业架构**的同构关系：架构设计即沙盘推演

## 真实库
- **pydantic**：CDP schema建模（对标Twilio Segment数据模型）
- **networkx + matplotlib**：架构依赖图建模与可视化（对标TOGAF/ArchiMate）
- **pandas**：行动研究迭代KPI分析
- **真实规范**：Segment Spec (https://segment.com/docs/spec/)


## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> 所有库（pydantic/networkx/matplotlib/pandas）均为本地可用库，不需要API Key。

In [ ]:
# !pip install pydantic networkx matplotlib pandas -q

import warnings
warnings.filterwarnings('ignore')

from datetime import datetime, timezone
from typing import Any, Optional
from pydantic import BaseModel, Field

import networkx as nx
import matplotlib
matplotlib.use('Agg')  # 非交互式后端，适合脚本运行
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np

print("环境就绪")
print(f"  pydantic: CDP schema建模")
print(f"  networkx: 架构依赖图")
print(f"  matplotlib: 架构图可视化")
print(f"  pandas: 行动研究KPI分析")


## 1. CDP身份层设计（对标Segment Identify Spec）

**CDP（客户数据平台）** 是AI原生架构的数据基础设施。本节用pydantic设计CDP四层schema的第一层--身份层(Identity)。

**Segment Identify Spec** 定义了三个核心字段：
- `userId`：已知用户的唯一标识（登录后）
- `anonymousId`：匿名用户的唯一标识（首次访问）
- `traits`：用户属性字典（年龄/性别/兴趣等自由格式）

**营销映射**：Identity层是CDP的基础--所有用户行为数据都挂在Identity上，Agent通过Identity查询用户画像。

参考：https://segment.com/docs/connections/spec/identify/


In [ ]:
# 1. CDP身份层设计 -- 用pydantic设计Identity模型（对标Segment Identify Spec）
# 参考：https://segment.com/docs/connections/spec/identify/

class Identity(BaseModel):
    """CDP身份层 -- 对标Segment Identify Spec

    Segment Identify定义了用户身份的三个核心字段：
    userId（已知用户标识）、anonymousId（匿名标识）、traits（用户属性）
    """
    user_id: str = Field(..., description="已知用户的唯一标识（登录后分配）")
    anonymous_id: str = Field(..., description="匿名用户的唯一标识（首次访问分配）")
    traits: dict[str, Any] = Field(default_factory=dict, description="用户属性字典（年龄/性别/兴趣等）")
    email: Optional[str] = Field(None, description="用户邮箱（可选）")
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc), description="身份创建时间")

# 实例化一个营销用户
identity = Identity(
    user_id="user_001",
    anonymous_id="anon_abc123",
    traits={
        "age": 28,
        "gender": "female",
        "interests": ["skincare", "fashion", "travel"],
        "membership_tier": "gold",
        "last_purchase_category": "beauty"
    },
    email="customer@example.com"
)
print("=== CDP身份层实例 ===")
print(identity.model_dump_json(indent=2))


## 2. CDP事件层设计（对标Segment Track Spec）

**Segment Track Spec** 定义了用户行为事件的三个核心字段：
- `userId`：触发事件的用户标识
- `event`：事件名称（如"Order Completed"、"Product Viewed"）
- `properties`：事件属性字典（订单金额/产品ID等）

**营销映射**：Event层记录用户与营销触点的所有交互--页面浏览、内容互动、购买行为。这些事件流是实时画像更新和Agent决策的数据源。

参考：https://segment.com/docs/connections/spec/track/


In [ ]:
# 2. CDP事件层设计 -- 用pydantic设计Event模型（对标Segment Track Spec）
# 参考：https://segment.com/docs/connections/spec/track/
from typing import Literal

# 营销事件类型（对标Segment Track的event字段）
MarketingEventType = Literal[
    "ProductViewed", "ProductAdded", "OrderCompleted",
    "EmailClicked", "AdImpression", "SearchPerformed",
    "ContentShared", "ReviewSubmitted"
]

class Event(BaseModel):
    """CDP事件层 -- 对标Segment Track Spec

    Segment Track定义了用户行为事件的三个核心字段：
    userId（用户标识）、event（事件名）、properties（事件属性）
    """
    user_id: str = Field(..., description="触发事件的用户标识")
    event_name: MarketingEventType = Field(..., description="事件名称（受Literal约束）")
    properties: dict[str, Any] = Field(default_factory=dict, description="事件属性字典")
    event_id: Optional[str] = Field(None, description="事件唯一ID（可选，自动生成）")
    timestamp: datetime = Field(default_factory=lambda: datetime.now(timezone.utc), description="事件时间戳")
    context: Optional[dict[str, Any]] = Field(None, description="事件上下文（设备/渠道等）")

# 实例化一个"用户浏览产品"事件
event = Event(
    user_id="user_001",
    event_name="ProductViewed",
    properties={
        "product_id": "SKU-2024-001",
        "product_name": "维C精华液",
        "category": "skincare",
        "price": 299.0,
        "currency": "CNY",
        "source": "mobile_app"
    },
    context={
        "device": "iPhone 15",
        "os": "iOS 18",
        "app_version": "3.2.1",
        "campaign_id": "camp_summer_2024"
    }
)
print("=== CDP事件层实例 ===")
print(event.model_dump_json(indent=2))


## 3. CDP分群层 + 画像层设计

**分群层(Segment)**：将用户按条件分组（如"高价值客户"、"流失风险用户"），支持营销Agent按分群执行差异化策略。

**画像层(Profile)**：整合Identity的traits、Event的聚合统计、Segment的归属信息，形成用户的完整360度画像。在AI原生架构中，Profile还包含向量化表示（embedding），支持语义级别的用户理解。

这两层共同构成CDP的"AI激活层"--Agent通过Profile API获取用户数据，通过Segment API获取目标人群。


In [ ]:
# 3. CDP分群层 + 画像层设计 -- 用pydantic设计Segment和Profile模型

class Segment(BaseModel):
    """CDP分群层 -- 将用户按条件分组，支持营销Agent按分群执行差异化策略"""
    segment_id: str = Field(..., description="分群唯一标识")
    name: str = Field(..., description="分群名称")
    criteria: str = Field(..., description="分群条件描述（如'近30天消费>500且复购>=2'）")
    user_ids: list[str] = Field(default_factory=list, description="分群内的用户ID列表")
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc), description="分群创建时间")

class Profile(BaseModel):
    """CDP画像层 -- 用户360度画像，含向量化表示（AI激活层）

    在AI原生架构中，Profile不仅存储结构化标签，还存储embedding向量，
    支持语义级别的用户理解和Agent调用。
    """
    user_id: str = Field(..., description="用户标识")
    traits: dict[str, Any] = Field(default_factory=dict, description="用户属性（来自Identity）")
    event_count: int = Field(default=0, description="用户累计事件数（行为聚合）")
    segments: list[str] = Field(default_factory=list, description="用户所属分群ID列表")
    embedding: list[float] = Field(default_factory=list, description="用户向量表示（embedding）")
    last_updated: datetime = Field(default_factory=lambda: datetime.now(timezone.utc), description="画像最后更新时间")

# 实例化"高价值客户"分群
segment = Segment(
    segment_id="seg_high_value",
    name="高价值客户",
    criteria="近30天消费>500元且复购>=2次且会员等级>=gold",
    user_ids=["user_001", "user_005", "user_012", "user_037", "user_089"]
)

# 实例化用户画像（含模拟embedding）
profile = Profile(
    user_id="user_001",
    traits={
        "age": 28, "gender": "female",
        "interests": ["skincare", "fashion", "travel"],
        "membership_tier": "gold"
    },
    event_count=147,
    segments=["seg_high_value", "seg_skincare_lover"],
    embedding=[0.12, -0.34, 0.56, 0.78, -0.23, 0.45, 0.67, -0.89],  # 模拟8维embedding
    last_updated=datetime.now(timezone.utc)
)

print("=== CDP分群层实例 ===")
print(segment.model_dump_json(indent=2))
print(f"\n分群用户数: {len(segment.user_ids)}")

print("\n=== CDP画像层实例 ===")
print(profile.model_dump_json(indent=2))
print(f"\n画像embedding维度: {len(profile.embedding)}")


## 4. 企业架构依赖图（TOGAF四层 + networkx）

用 **networkx DiGraph** 建模企业AI架构的组件依赖关系，对标 **TOGAF/ArchiMate** 的四层架构域：

| TOGAF架构域 | AI原生架构组件 |
|-------------|---------------|
| 业务架构 | MarketingCampaign, CustomerJourney |
| 应用架构 | InsightAgent, ContentAgent, PlacementAgent, AnalyticsAgent, CoordinatorAgent |
| 数据架构 | CDP(Identity/Event/Segment/Profile), VectorDB, KnowledgeGraph |
| 技术架构 | LLMService, RAGEngine, DataPipeline, InferenceServer |

**任务**：构建架构依赖图，计算节点数/边数，分析关键依赖路径（如"数据层故障如何影响业务层"）。

参考：https://www.opengroup.org/capabilities/togaf


In [ ]:
# 4. 企业架构依赖图 -- 用networkx DiGraph建模TOGAF四层架构
# 参考：TOGAF四层架构域 https://www.opengroup.org/capabilities/togaf

G = nx.DiGraph()

# 技术架构层 (Technology Layer)
tech_nodes = ["LLMService", "RAGEngine", "DataPipeline", "InferenceServer"]
for n in tech_nodes:
    G.add_node(n, layer="technology")

# 数据架构层 (Data Layer)
data_nodes = ["CDP_Identity", "CDP_Event", "CDP_Segment", "CDP_Profile",
              "VectorDB", "KnowledgeGraph"]
for n in data_nodes:
    G.add_node(n, layer="data")

# 应用架构层 (Application Layer)
app_nodes = ["InsightAgent", "ContentAgent", "PlacementAgent",
             "AnalyticsAgent", "CoordinatorAgent"]
for n in app_nodes:
    G.add_node(n, layer="application")

# 业务架构层 (Business Layer)
biz_nodes = ["MarketingCampaign", "CustomerJourney"]
for n in biz_nodes:
    G.add_node(n, layer="business")

# 添加依赖边（A -> B 表示 B 依赖 A）
# 技术层内部
G.add_edge("DataPipeline", "LLMService")
G.add_edge("DataPipeline", "RAGEngine")
G.add_edge("InferenceServer", "LLMService")

# 数据层 -> 技术层
G.add_edge("DataPipeline", "CDP_Event")
G.add_edge("DataPipeline", "CDP_Identity")
G.add_edge("VectorDB", "CDP_Profile")
G.add_edge("KnowledgeGraph", "CDP_Profile")

# 数据层内部
G.add_edge("CDP_Identity", "CDP_Profile")
G.add_edge("CDP_Event", "CDP_Profile")
G.add_edge("CDP_Profile", "CDP_Segment")

# 应用层 -> 数据层
G.add_edge("CDP_Profile", "InsightAgent")
G.add_edge("CDP_Segment", "InsightAgent")
G.add_edge("CDP_Profile", "ContentAgent")
G.add_edge("CDP_Segment", "PlacementAgent")
G.add_edge("CDP_Event", "AnalyticsAgent")
G.add_edge("CDP_Profile", "AnalyticsAgent")

# 应用层 -> 技术层
G.add_edge("LLMService", "InsightAgent")
G.add_edge("RAGEngine", "ContentAgent")
G.add_edge("LLMService", "ContentAgent")
G.add_edge("LLMService", "PlacementAgent")
G.add_edge("LLMService", "AnalyticsAgent")

# 应用层内部
G.add_edge("InsightAgent", "CoordinatorAgent")
G.add_edge("ContentAgent", "CoordinatorAgent")
G.add_edge("PlacementAgent", "CoordinatorAgent")
G.add_edge("AnalyticsAgent", "CoordinatorAgent")

# 业务层 -> 应用层
G.add_edge("CoordinatorAgent", "MarketingCampaign")
G.add_edge("CoordinatorAgent", "CustomerJourney")

print("=== 企业架构依赖图（TOGAF四层）===")
print(f"节点数: {G.number_of_nodes()}")
print(f"边数: {G.number_of_edges()}")
print(f"\n各层节点分布:")
for layer in ["technology", "data", "application", "business"]:
    nodes = [n for n in G.nodes if G.nodes[n].get("layer") == layer]
    print(f"  {layer}: {nodes}")

# 关键依赖路径分析
print(f"\n关键依赖路径（DataPipeline -> MarketingCampaign）:")
path = nx.shortest_path(G, "DataPipeline", "MarketingCampaign")
print(f"  {' -> '.join(path)}")
print(f"  路径长度: {len(path) - 1} 跳")

print(f"\n关键依赖路径（CDP_Identity -> CustomerJourney）:")
path2 = nx.shortest_path(G, "CDP_Identity", "CustomerJourney")
print(f"  {' -> '.join(path2)}")
print(f"  路径长度: {len(path2) - 1} 跳")

# 分析最脆弱的节点（被依赖最多的节点）
in_degree = dict(G.in_degree())
top_critical = sorted(in_degree.items(), key=lambda x: x[1], reverse=True)[:5]
print(f"\n被依赖最多的节点（Top 5，单点故障风险）:")
for node, deg in top_critical:
    print(f"  {node}: 被依赖 {deg} 次")


## 5. CDP数据流图可视化

用 **networkx + matplotlib** 可视化CDP在营销中心的数据流：

```
数据源(Website/App/CRM)
  ↓ Identify/Track API
CDP身份层+事件层
  ↓ 实时流处理
CDP画像层+分群层
  ↓ Profile/Segment API
Agent编排层(洞察/内容/投放/分析)
  ↓ 营销动作
触达渠道(邮件/短信/广告)
```

**任务**：构建CDP数据流图，用matplotlib绘制可视化，标注数据流向和组件类型。


In [ ]:
# 5. CDP数据流图可视化 -- 用networkx+matplotlib绘制
flow_g = nx.DiGraph()

# 数据源
sources = ["Website", "MobileApp", "CRM"]
for n in sources:
    flow_g.add_node(n, node_type="source")

# CDP四层
cdp_nodes = ["Identify_API", "Track_API", "Identity_Layer", "Event_Layer",
             "Profile_Layer", "Segment_Layer"]
for n in cdp_nodes:
    flow_g.add_node(n, node_type="cdp")

# Agent层
agent_nodes = ["InsightAgent", "ContentAgent", "PlacementAgent"]
for n in agent_nodes:
    flow_g.add_node(n, node_type="agent")

# 触达渠道
channels = ["Email", "SMS", "Ad_Platform"]
for n in channels:
    flow_g.add_node(n, node_type="channel")

# 数据流边
# 数据源 -> CDP API
flow_g.add_edge("Website", "Identify_API")
flow_g.add_edge("Website", "Track_API")
flow_g.add_edge("MobileApp", "Identify_API")
flow_g.add_edge("MobileApp", "Track_API")
flow_g.add_edge("CRM", "Identify_API")

# API -> CDP层
flow_g.add_edge("Identify_API", "Identity_Layer")
flow_g.add_edge("Track_API", "Event_Layer")
flow_g.add_edge("Identity_Layer", "Profile_Layer")
flow_g.add_edge("Event_Layer", "Profile_Layer")
flow_g.add_edge("Profile_Layer", "Segment_Layer")

# CDP -> Agent
flow_g.add_edge("Profile_Layer", "InsightAgent")
flow_g.add_edge("Segment_Layer", "ContentAgent")
flow_g.add_edge("Segment_Layer", "PlacementAgent")
flow_g.add_edge("Profile_Layer", "PlacementAgent")

# Agent -> 渠道
flow_g.add_edge("ContentAgent", "Email")
flow_g.add_edge("ContentAgent", "SMS")
flow_g.add_edge("PlacementAgent", "Ad_Platform")

# 可视化
fig, ax = plt.subplots(1, 1, figsize=(14, 8))

# 自定义分层布局
pos = {}
# 数据源在左
for i, n in enumerate(sources):
    pos[n] = (0, 3 - i * 1.2)
# CDP API在中左
pos["Identify_API"] = (2, 2.5)
pos["Track_API"] = (2, 0.5)
# CDP层在中
for i, n in enumerate(["Identity_Layer", "Event_Layer"]):
    pos[n] = (4, 3 - i * 2)
for i, n in enumerate(["Profile_Layer", "Segment_Layer"]):
    pos[n] = (6, 3 - i * 2)
# Agent在中右
for i, n in enumerate(agent_nodes):
    pos[n] = (8, 3 - i * 1.2)
# 渠道在右
for i, n in enumerate(channels):
    pos[n] = (10, 2.5 - i * 1.5)

# 颜色映射
color_map = {"source": "#4ECDC4", "cdp": "#45B7D1", "agent": "#FFA07A", "channel": "#98D8C8"}
node_colors = [color_map[flow_g.nodes[n].get("node_type", "source")] for n in flow_g.nodes]

nx.draw(flow_g, pos, with_labels=True, node_color=node_colors,
        node_size=2200, font_size=8, font_weight='bold',
        arrows=True, arrowsize=20, edge_color='#888888',
        ax=ax)
ax.set_title("CDP数据流图：数据源 -> CDP四层 -> Agent编排 -> 触达渠道", fontsize=13, fontweight='bold')

# 添加图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#4ECDC4", label="数据源 (Source)"),
    Patch(facecolor="#45B7D1", label="CDP层 (Identity/Event/Profile/Segment)"),
    Patch(facecolor="#FFA07A", label="Agent编排层"),
    Patch(facecolor="#98D8C8", label="触达渠道 (Channel)"),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig("cdp_data_flow.png", dpi=150, bbox_inches='tight')
plt.show()

print("=== CDP数据流图 ===")
print(f"节点数: {flow_g.number_of_nodes()}")
print(f"边数: {flow_g.number_of_edges()}")
print(f"\n节点类型分布:")
for nt in ["source", "cdp", "agent", "channel"]:
    count = sum(1 for n in flow_g.nodes if flow_g.nodes[n].get("node_type") == nt)
    print(f"  {nt}: {count}个节点")

# 关键数据流路径
print(f"\n关键数据流（Website -> Ad_Platform）:")
flow_path = nx.shortest_path(flow_g, "Website", "Ad_Platform")
print(f"  {' -> '.join(flow_path)}")


## 6. 行动研究迭代分析（Plan/Act/Observe/Reflect）

用 **pandas** 分析行动研究的迭代循环数据。行动研究（Action Research）采用Susman & Evered (1978)的五步螺旋：诊断->规划->行动->评估->反思。

**数据说明**：以下KPI数据基于真实行动研究文献报告的改善幅度区间构建（非单一案例精确数据）：
- 决策时间：AI部署后降低30%-60%
- 决策质量：提升1.0-2.5分（10分制）
- AI使用率：从10%->70%（随迭代轮次增长）
- 团队满意度：首轮下降0.3-0.5（学习曲线），后续回升0.5-1.0

来源：Susman & Evered (1978); Kemmis et al. (2014); Coughlan & Coghlan (2002)

**任务**：加载行动研究迭代数据，分析各轮KPI变化趋势，计算改善幅度。


In [ ]:
# 6. 行动研究迭代分析 -- 用pandas分析Plan/Act/Observe/Reflect多轮KPI
# KPI改善幅度参考真实行动研究文献：
#   Susman & Evered (1978) https://doi.org/10.1016/0360-1315(78)90013-0
#   Kemmis et al. (2014) https://doi.org/10.1080/09650792.2014.922340
#   Coughlan & Coghlan (2002) https://doi.org/10.1080/09650790210100233

# 行动研究4轮迭代数据（基于真实文献报告的改善幅度区间）
ar_data = [
    # Round 0: 基线（AI部署前）
    {"round": 0, "phase": "Diagnose", "decision_time_min": 45.0,
     "decision_quality": 6.0, "ai_usage_rate": 0.0, "team_satisfaction": 3.8},
    # Round 1: 规划+首轮行动（AI初步部署，学习曲线导致满意度下降）
    {"round": 1, "phase": "Plan+Act", "decision_time_min": 38.0,
     "decision_quality": 6.5, "ai_usage_rate": 12.0, "team_satisfaction": 3.4},
    # Round 2: 评估+反思后调整（AI使用率上升，效率提升）
    {"round": 2, "phase": "Observe+Reflect", "decision_time_min": 28.0,
     "decision_quality": 7.8, "ai_usage_rate": 35.0, "team_satisfaction": 3.9},
    # Round 3: 第二轮行动（Agent全面集成，效果显著）
    {"round": 3, "phase": "Act", "decision_time_min": 19.0,
     "decision_quality": 8.5, "ai_usage_rate": 58.0, "team_satisfaction": 4.3},
    # Round 4: 评估+反思（AI使用率达标，满意度回升超过基线）
    {"round": 4, "phase": "Evaluate+Reflect", "decision_time_min": 15.0,
     "decision_quality": 9.2, "ai_usage_rate": 72.0, "team_satisfaction": 4.6},
]

ar_df = pd.DataFrame(ar_data)
print("=== 行动研究迭代数据（4轮 + 基线）===")
print(ar_df.to_string(index=False))

# 计算每轮相对基线(Round 0)的改善幅度
baseline = ar_df[ar_df["round"] == 0].iloc[0]
improvement_rows = []
for _, row in ar_df.iterrows():
    r = row["round"]
    if r == 0:
        continue
    improvement_rows.append({
        "round": r,
        "phase": row["phase"],
        "time_change_pct": round((row["decision_time_min"] - baseline["decision_time_min"]) / baseline["decision_time_min"] * 100, 1),
        "quality_change": round(row["decision_quality"] - baseline["decision_quality"], 1),
        "ai_usage_change_pct": round(row["ai_usage_rate"] - baseline["ai_usage_rate"], 1),
        "satisfaction_change": round(row["team_satisfaction"] - baseline["team_satisfaction"], 1),
    })

improvement_df = pd.DataFrame(improvement_rows)
print("\n=== 各轮相对基线(Round 0)的改善幅度 ===")
print(improvement_df.to_string(index=False))

# 趋势分析
print("\n=== 趋势分析 ===")
print(f"决策时间: 从 {baseline['decision_time_min']:.0f}分钟 降至 {ar_df.iloc[-1]['decision_time_min']:.0f}分钟 "
      f"(降低 {abs(improvement_df.iloc[-1]['time_change_pct']):.1f}%)")
print(f"决策质量: 从 {baseline['decision_quality']:.1f} 升至 {ar_df.iloc[-1]['decision_quality']:.1f} "
      f"(提升 {improvement_df.iloc[-1]['quality_change']:.1f}分)")
print(f"AI使用率: 从 {baseline['ai_usage_rate']:.0f}% 升至 {ar_df.iloc[-1]['ai_usage_rate']:.0f}%")
print(f"团队满意度: 从 {baseline['team_satisfaction']:.1f} 升至 {ar_df.iloc[-1]['team_satisfaction']:.1f} "
      f"(首轮下降至 {ar_df.iloc[1]['team_satisfaction']:.1f} 后回升)")

# 哪轮改善最大
time_improvements = improvement_df["time_change_pct"].tolist()
max_round = improvement_df.loc[improvement_df["time_change_pct"].idxmin(), "round"]
print(f"\n决策时间改善最大的是 Round {max_round}（降低 {abs(min(time_improvements)):.1f}%）")
print(f"改善最显著的KPI: AI使用率（从0%到72%，增长72个百分点）")

# 可视化KPI趋势
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("行动研究迭代KPI趋势（Susman & Evered五步螺旋）", fontsize=14, fontweight='bold')

ar_df.plot(x="round", y="decision_time_min", ax=axes[0, 0], marker='o', color='#E74C3C', legend=False)
axes[0, 0].set_title("决策时间（分钟）- 越低越好")
axes[0, 0].set_xlabel("迭代轮次")
axes[0, 0].axhline(y=baseline["decision_time_min"], color='gray', linestyle='--', alpha=0.5, label='基线')

ar_df.plot(x="round", y="decision_quality", ax=axes[0, 1], marker='s', color='#2ECC71', legend=False)
axes[0, 1].set_title("决策质量（1-10分）- 越高越好")
axes[0, 1].set_xlabel("迭代轮次")

ar_df.plot(x="round", y="ai_usage_rate", ax=axes[1, 0], marker='^', color='#3498DB', legend=False)
axes[1, 0].set_title("AI使用率（%）- 随迭代增长")
axes[1, 0].set_xlabel("迭代轮次")

ar_df.plot(x="round", y="team_satisfaction", ax=axes[1, 1], marker='D', color='#9B59B6', legend=False)
axes[1, 1].set_title("团队满意度（1-5分）- 先降后升（学习曲线）")
axes[1, 1].set_xlabel("迭代轮次")
axes[1, 1].axhline(y=baseline["team_satisfaction"], color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig("action_research_kpi.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n图表已保存: action_research_kpi.png")


## 7. 反思与前沿

### 反思问题
1. 你的CDP schema设计是否覆盖了营销Agent需要的所有数据？缺什么？
2. 架构依赖图中哪条路径最脆弱？如果CDP故障，哪些Agent会受影响？
3. 行动研究的KPI改善幅度是否符合预期？哪轮改善最大？为什么？
4. 用天道推演视角分析：你在架构设计时推演了几个方案？每个方案的3层未来走向是什么？

### 2026前沿：天道推演×企业架构 + DSR + 可复现研究

**天道推演×企业架构**：企业架构设计本质上是对组织的沙盘推演--架构师在意识中构建多个架构方案的平行世界，模拟其未来走向，选择最优路径。天道推演的五项能力（局势感知/因果链追踪/沙盘模拟/概率评估/最优路径推荐）与企业架构设计的五个阶段（现状审计/依赖分析/方案模拟/风险评估/选型推荐）同构。

**多Agent仿真×架构验证**：2026年前沿趋势是用多Agent仿真验证架构设计--在部署真实系统前，先用多Agent仿真模拟架构运行情况，预测消息传递延迟、资源竞争、故障传播路径。

**DSR + 可复现研究**：企业架构设计作为DSR artifact，行动研究作为评估方法。你的架构依赖图（networkx）+ CDP schema（pydantic）+ 行动研究KPI（pandas）全部用代码定义，他人可独立复现。

### 天道推演×企业架构 同构映射表

| 天道推演能力 | 企业架构设计对应 | 本Day上机对应 |
|-------------|----------------|-------------|
| 局势感知 | 现有架构审计 | CDP schema现状分析 |
| 因果链追踪 | 架构依赖图分析 | networkx DAG + shortest_path |
| 沙盘模拟(3层) | 多架构方案并行模拟 | CDP数据流图可视化 |
| 概率评估 | 方案风险/成本/收益概率 | 行动研究KPI不确定性 |
| 最优路径推荐 | 架构选型建议 | 营销中心参考架构 |

> 深入阅读见 `reading.md` 的TOGAF、行动研究、DSR条目。
